# Matrix Decompositions

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 01.04, 01.06, 01.07 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/08_matrix_decompositions.ipynb)

---

## 🎯 Learning Objective

Survey the major matrix factorizations — QR, Cholesky, LU, Schur — and learn which decomposition to reach for depending on the structure of your problem.

---

## 📐 Theory

Different matrix structures call for different factorizations. A quick field guide:

**QR decomposition**: $A = QR$, with $Q$ orthogonal and $R$ upper triangular. Built via
Gram-Schmidt orthogonalization (or the numerically sturdier Householder reflections). QR is
the standard, numerically stable way to solve least-squares problems without ever forming
$A^\top A$ (which, as in the normal equations of
[Notebook 01.04](04_systems_of_equations.ipynb), can lose precision).

**Cholesky decomposition**: for symmetric positive-definite $A$, $A = LL^\top$ with $L$ lower
triangular. It's roughly twice as fast as general-purpose factorizations *because* it exploits
positive-definiteness, and it's the standard way to sample from a multivariate Gaussian with
covariance $A$.

**LU decomposition**: $A = LU$ (or $PA = LU$ with a row-permutation $P$ for numerical
stability), the "packaged" output of Gaussian elimination introduced in
[Notebook 01.04](04_systems_of_equations.ipynb).

**Schur decomposition**: $A = QTQ^\top$ (or with complex entries, $A=UTU^*$) where $T$ is upper
triangular and $Q$/$U$ is orthogonal/unitary — this always exists, even for non-symmetric $A$
where eigendecomposition might not, making it the general-purpose workhorse many library
eigenvalue routines actually compute internally.

### Choosing a decomposition

| Situation | Reach for |
|---|---|
| Solving $A\mathbf{x}=\mathbf{b}$ once | LU |
| Solving with many different $\mathbf{b}$, or least-squares | QR |
| $A$ symmetric positive-definite (e.g. a covariance matrix) | Cholesky |
| Need eigenvalues, $A$ not symmetric | Schur |

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as sla
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Visualizing the factor matrices themselves as heatmaps shows their structure directly: QR's
$R$ is upper triangular, Cholesky's $L$ is lower triangular, and both are "sparser looking"
than the original dense matrix.

In [ ]:
# Heatmaps of the factor matrices reveal their triangular structure directly
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as sla

rng = np.random.default_rng(0)
B = rng.standard_normal((5, 5))
A_spd = B @ B.T + 5 * np.eye(5)   # guaranteed symmetric positive-definite

Q, R = np.linalg.qr(B)
L_chol = np.linalg.cholesky(A_spd)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (title, M) in zip(axes, [("R (upper triangular, from QR)", R),
                                   ("L (lower triangular, from Cholesky)", L_chol),
                                   ("Original SPD matrix A", A_spd)]):
    im = ax.imshow(M, cmap="coolwarm")
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

Gram-Schmidt orthogonalization builds QR column by column, and a simple algorithm builds
Cholesky row by row — both checked against SciPy/NumPy.

In [ ]:
# QR via Gram-Schmidt, and Cholesky via its defining recursive formula
import numpy as np

def qr_gram_schmidt(A, rank_tol=1e-10):
    """Classical Gram-Schmidt. Each column's remaining component v, after subtracting off
    everything already covered by earlier columns, is normalized by R[j,j]=||v||. If the
    input's columns are linearly dependent (or numerically close to it), v shrinks toward
    the zero vector, R[j,j] shrinks toward 0, and Q[:,j]=v/R[j,j] would silently divide by
    (near) zero, returning nan/inf with no indication of WHY. We guard that division
    explicitly and raise a clear error instead -- "rank-deficient input" is the actual
    problem, and a silent nan is a much worse failure mode than a raised exception.
    (Classical Gram-Schmidt is also known to lose orthogonality for ill-conditioned inputs
    even short of exact singularity -- the Theory section already notes that Householder
    reflections are the numerically sturdier choice used in practice, e.g. np.linalg.qr.)"""
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i, j] = Q[:, i] @ A[:, j]
            v -= R[i, j] * Q[:, i]           # subtract off the component already covered by earlier columns
        R[j, j] = np.linalg.norm(v)
        if R[j, j] < rank_tol:
            raise np.linalg.LinAlgError(
                f"Column {j} is (numerically) linearly dependent on the previous columns "
                f"(||residual||={R[j, j]:.2e} < rank_tol={rank_tol:.0e}) -- A does not have "
                f"full column rank, so QR via Gram-Schmidt is not well-defined here."
            )
        Q[:, j] = v / R[j, j]
    return Q, R

def cholesky_from_scratch(A):
    """Cholesky's defining recursive formula ASSUMES A is symmetric positive-definite: that
    assumption is exactly what guarantees every A[i,i]-s (the diagonal case) stays positive,
    so sqrt() never sees a negative argument. If A is not actually SPD, that guarantee is
    gone -- np.sqrt of a negative number silently returns nan (with a RuntimeWarning)
    instead of a clear "not positive-definite" error, and a zero/near-zero L[j,j] on the
    next row would then also divide silently. We check the precondition up front instead."""
    n = A.shape[0]
    if not np.allclose(A, A.T):
        raise ValueError("Cholesky requires a SYMMETRIC matrix; A != A^T here.")
    L = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1):
            s = sum(L[i, k] * L[j, k] for k in range(j))
            if i == j:
                diag_term = A[i, i] - s
                if diag_term <= 0:
                    raise np.linalg.LinAlgError(
                        f"A is not positive-definite: the diagonal term at i={i} is "
                        f"{diag_term:.2e} <= 0, so sqrt() would be applied to a "
                        f"non-positive number (this is precisely what positive-definiteness "
                        f"rules out; np.linalg.cholesky raises the same LinAlgError here)."
                    )
                L[i, j] = np.sqrt(diag_term)
            else:
                L[i, j] = (A[i, j] - s) / L[j, j]
    return L

rng = np.random.default_rng(1)
B = rng.standard_normal((5, 4))
Q_scratch, R_scratch = qr_gram_schmidt(B)
Q_np, R_np = np.linalg.qr(B)
print("Gram-Schmidt QR reconstructs B:  ", np.allclose(Q_scratch @ R_scratch, B))
print("Q is orthogonal (Q^T Q = I):      ", np.allclose(Q_scratch.T @ Q_scratch, np.eye(4)))

C = rng.standard_normal((4, 4))
A_spd = C @ C.T + 4 * np.eye(4)
L_scratch = cholesky_from_scratch(A_spd)
L_np = np.linalg.cholesky(A_spd)
print("\nCholesky from scratch matches np.linalg.cholesky:", np.allclose(L_scratch, L_np))
print("L @ L^T reconstructs A:                            ", np.allclose(L_scratch @ L_scratch.T, A_spd))

# --- Regression test: qr_gram_schmidt on a RANK-DEFICIENT input (3rd column = col0+col1) ---
print("\n--- Regression test: qr_gram_schmidt on a rank-deficient matrix ---")
B_deficient = rng.standard_normal((5, 3))
B_deficient[:, 2] = B_deficient[:, 0] + B_deficient[:, 1]   # exactly linearly dependent
try:
    qr_gram_schmidt(B_deficient)
    print("FAILED to detect rank deficiency (this should not happen)")
except np.linalg.LinAlgError as e:
    print(f"Correctly raised LinAlgError instead of returning silent nan/inf: {e}")

# --- Regression test: cholesky_from_scratch on a matrix that is NOT positive-definite ---
print("\n--- Regression test: cholesky_from_scratch on a non-positive-definite matrix ---")
A_not_pd = np.array([[1.0, 2.0], [2.0, 1.0]])   # symmetric, but eigenvalues are 3 and -1
print(f"Eigenvalues of A_not_pd: {np.round(np.linalg.eigvalsh(A_not_pd), 4)} (one is negative -> not PD)")
try:
    cholesky_from_scratch(A_not_pd)
    print("FAILED to detect non-positive-definiteness (this should not happen)")
except np.linalg.LinAlgError as e:
    print(f"Correctly raised LinAlgError instead of returning silent nan: {e}")
try:
    np.linalg.cholesky(A_not_pd)
except np.linalg.LinAlgError:
    print("np.linalg.cholesky agrees: it also raises LinAlgError on this input.")

## 🤖 Why This Matters for AI

**Cholesky decomposition is THE standard way to sample from a multivariate Gaussian.** Given a
covariance matrix $\Sigma = LL^\top$, you can turn independent standard-normal samples $z$
into correctly-correlated samples via $\mathbf{x} = \mu + Lz$. This exact trick underlies the
**reparameterization trick** in Variational Autoencoders, which is what makes it possible to
backpropagate through a random sampling step at all.

Why Cholesky specifically? It costs $O(d^3/3)$ — half a full eigendecomposition — and produces a triangular factor that is both fast to solve against and numerically stable. `torch.distributions.MultivariateNormal` uses `torch.linalg.cholesky` internally to sample from any covariance matrix, and Gaussian process libraries like `gpytorch` use it to solve the $n \times n$ kernel system.

---


In [ ]:
# The reparameterization trick: sampling a correlated Gaussian via Cholesky, differentiably
import numpy as np

rng = np.random.default_rng(0)
mu = np.array([2.0, -1.0])
Sigma = np.array([[2.0, 1.2], [1.2, 1.5]])    # target covariance -- off-diagonal means correlated dims
L = np.linalg.cholesky(Sigma)

n_samples = 5000
z = rng.standard_normal((n_samples, 2))        # cheap to sample: independent standard normals
x = mu + z @ L.T                                # x = mu + L @ z  (the reparameterization trick)

print("Target covariance:\n", np.round(Sigma, 3))
print("Empirical covariance of samples:\n", np.round(np.cov(x.T), 3))
print("\nTarget mean:", mu, " Empirical mean:", np.round(x.mean(axis=0), 3))
print("\nCrucially: x = mu + L @ z is a DIFFERENTIABLE function of mu and L (hence of Sigma).")
print("This is exactly why VAEs can backprop through a 'random' sampling step during training.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $A=\begin{pmatrix}4&2\\2&5\end{pmatrix}$, compute its Cholesky factor $L$ by hand using
   the recursive formula, then verify with `np.linalg.cholesky`. As a sanity check, confirm
   $LL^\top$ reconstructs $A$ exactly.

<details>
<summary>💡 Solution</summary>

$L_{11}=\sqrt{A_{11}}=\sqrt4=2$. $L_{21}=A_{21}/L_{11}=2/2=1$.
$L_{22}=\sqrt{A_{22}-L_{21}^2}=\sqrt{5-1}=2$. So
$$L=\begin{pmatrix}2&0\\1&2\end{pmatrix},$$
matching `np.linalg.cholesky(A)` exactly. Check: $LL^\top=\begin{pmatrix}4&2\\2&5\end{pmatrix}=A$
exactly, as it must for any correctly-computed Cholesky factor.

</details>

### 💭 UNDERSTAND
2. `cholesky_from_scratch` explicitly raises an error whenever it hits a diagonal term
   $\le0$. Without running any code, predict whether it succeeds or raises an error on each of
   these three symmetric matrices, and *why*: (a) $M_1=\begin{pmatrix}5&1&0\\1&4&1\\0&1&3
   \end{pmatrix}$, (b) $M_2=\begin{pmatrix}1&2\\2&1\end{pmatrix}$, (c)
   $M_3=\begin{pmatrix}1&1\\1&1\end{pmatrix}$. Then run it to check.

<details>
<summary>💡 Solution</summary>

$M_1$ succeeds: its eigenvalues ($\approx2.27,4.00,5.73$) are all strictly positive, so it's
positive-definite. $M_2$ fails: its eigenvalues are $3$ and $-1$ — one negative, so it's not
even positive-*semi*-definite, and the function correctly raises at $i=1$ with a negative
diagonal term ($-3.0$). $M_3$ is the interesting edge case: its eigenvalues are $2$ and exactly
$0$ — it *is* positive-**semi**-definite (no negative eigenvalue) but not strictly
positive-*definite* (a zero eigenvalue exists), and the function still raises (diagonal term
exactly $0.0$ at $i=1$), because the Cholesky factorization the Theory section describes is
only guaranteed for strictly positive-definite matrices — a zero eigenvalue means $L_{22}$
would have to be exactly $0$, which the function's `diag_term <= 0` check (deliberately using
$\le$, not $<$) correctly refuses to accept as a valid strictly-positive-definite input, matching
what `np.linalg.cholesky` also refuses.

</details>

### ✏️ DERIVE
3. Derive the Cholesky recursion $L_{ii}=\sqrt{A_{ii}-\sum_{k<i}L_{ik}^2}$ and
   $L_{ij}=\big(A_{ij}-\sum_{k<j}L_{ik}L_{jk}\big)/L_{jj}$ (for $i>j$) directly from the
   defining equation $A=LL^\top$, i.e. $A_{ij}=\sum_k L_{ik}L_{jk}$, using the fact that $L$ is
   lower triangular (so $L_{ik}=0$ whenever $k>i$).

<details>
<summary>💡 Solution outline</summary>

Since $L$ is lower triangular, $L_{ik}=0$ for $k>i$ and $L_{jk}=0$ for $k>j$, so the sum
$A_{ij}=\sum_k L_{ik}L_{jk}$ only has nonzero terms for $k\le\min(i,j)$. **Diagonal case
($i=j$):** the sum runs over $k\le i$, and the $k=i$ term is $L_{ii}\cdot L_{ii}=L_{ii}^2$
(the only term not already determined by earlier rows), so
$$A_{ii}=\sum_{k<i}L_{ik}^2+L_{ii}^2 \;\;\Longrightarrow\;\; L_{ii}=\sqrt{A_{ii}-\sum_{k<i}L_{ik}^2}.$$
**Off-diagonal case ($i>j$):** the sum runs over $k\le j$ (since $j<i$), and the $k=j$ term is
$L_{ij}L_{jj}$ (the one unknown, since all $L_{ik}$ for $k<j$ and $L_{jk}$ for $k<j$ were
already computed in earlier rows/columns):
$$A_{ij}=\sum_{k<j}L_{ik}L_{jk}+L_{ij}L_{jj} \;\;\Longrightarrow\;\;
L_{ij}=\frac{A_{ij}-\sum_{k<j}L_{ik}L_{jk}}{L_{jj}}.$$
This is exactly `cholesky_from_scratch`'s two branches. Numeric check on a random $4\times4$
SPD matrix: computing entry $A_{2,1}$ directly and via $\sum_{k\le1}L_{2,k}L_{1,k}$ from the
already-built $L$ agree exactly, confirming the recursion reconstructs every entry of $A$, not
just the ones used to derive it.

</details>

### 🐍 IMPLEMENT
4. Use `qr_gram_schmidt` to factor a random $6\times3$ matrix, and use the resulting $Q,R$ to
   solve a least-squares problem $A\mathbf{x}\approx\mathbf{b}$ via back-substitution on
   $R\mathbf{x}=Q^\top\mathbf{b}$.

<details>
<summary>✅ How to know you're right</summary>

Your $\mathbf{x}$ must match `np.linalg.lstsq(A, b)`'s solution to floating-point precision, and
the two approaches' residual norms $\|A\mathbf{x}-\mathbf{b}\|$ must match too — QR-based
least-squares and `lstsq` are solving the exact same minimization problem by different
numerically stable routes, so any mismatch beyond floating-point noise means a bug in your
back-substitution (a common one: solving $R\mathbf{x}=Q^\top\mathbf{b}$ top-down instead of
bottom-up — $R$ is upper triangular, so back-substitution must start from the *last* row).

</details>

### 🤖 APPLY
5. Gaussian process (GP) regression needs to sample from a high-dimensional multivariate
   Gaussian whose covariance is a kernel matrix, $K_{ij}=k(x_i,x_j)$ for an RBF kernel over a
   grid of 30 points. Build $K$, add a small "jitter" term ($10^{-6}I$) to its diagonal, and use
   this notebook's own Cholesky-based reparameterization trick to draw many samples from
   $\mathcal{N}(\mathbf{0}, K)$. Verify the empirical covariance of your samples matches $K$.

<details>
<summary>✅ What you should observe</summary>

Without the jitter term, `np.linalg.cholesky(K)` should actually *fail* on the raw RBF kernel
matrix here — its smallest eigenvalue comes out as a tiny negative number
($\approx-1.8\times10^{-16}$, pure floating-point noise around a true eigenvalue that should be
exactly $0$ but isn't representable exactly), which is exactly what "not positive-definite"
means numerically even though the RBF kernel is mathematically guaranteed positive-*semi*-
definite. Adding jitter ($10^{-6}I$, i.e. raising every eigenvalue by $10^{-6}$) fixes this
cleanly, and 20,000 samples drawn via $\mathbf{f}=L\mathbf{z}$ should reproduce the jittered $K$
closely (maximum absolute entry-wise difference on the order of $0.02$, shrinking with more
samples, since this is a Monte Carlo estimate) and a sample mean close to zero. This is the
exact same reparameterization mechanism as the notebook's own VAE example, just with a
much larger (and structured, kernel-derived) covariance matrix — and the jitter trick itself is
worth noticing: it's a tiny, deliberate loosening of "exactly positive-semi-definite" to "safely
positive-definite," trading a negligible amount of accuracy for numerical reliability, the same
kind of tradeoff `01.06`'s condition-number discussion and `01.04`'s ridge-regularization
exercise both make in different contexts.

</details>

### 🚀 CHALLENGE
6. Compute the Schur decomposition of a random non-symmetric $4\times4$ matrix constructed to
   have a genuine complex-conjugate eigenvalue pair (e.g. build it as a similarity transform of
   a block-diagonal matrix containing a 2D rotation-like block and two distinct real
   eigenvalues), using `scipy.linalg.schur`. Confirm the eigenvalues of the resulting
   (quasi-triangular) $T$ match `np.linalg.eigvals(A)` exactly, and inspect $T$'s structure
   around the complex-conjugate pair. Explain why Schur's decomposition exists here even though
   this matrix has no valid *orthogonal* eigendecomposition.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer confirms the eigenvalues match exactly between `np.linalg.eigvals(A)` and
`np.linalg.eigvals(T)` (as they must — $T$ and $A$ are similar matrices, via the orthogonal $Z$
with $A=ZTZ^\top$, and similar matrices always share eigenvalues), and $Z^\top Z=I$
confirms $Z$'s orthogonality directly. Inspecting $T$'s structure, a strong answer notices it is
only *quasi*-triangular: the real eigenvalues (here, $2$ and $3$) sit as ordinary $1\times1$
diagonal blocks with zeros below them, but the complex-conjugate pair sits in a $2\times2$ block
straddling the diagonal (with a nonzero entry directly below the diagonal at that position) —
this $2\times2$ block's own eigenvalues (computed from just those 4 entries) are exactly the
complex-conjugate pair, which is *how* Schur represents a complex eigenvalue using only real
arithmetic throughout $T$ and $Z$. The deeper point: Schur's decomposition exists for literally
*any* square matrix (real or complex, symmetric or not, defective or not) because — unlike
eigendecomposition, which needs $n$ linearly independent eigenvectors to even be well-defined —
it only ever needs *some* orthonormal basis in which $A$'s action becomes triangular, and such a
basis can always be built one Krylov-like step at a time; a matrix with a complex-conjugate
eigenvalue pair simply cannot be triangularized using a real, orthogonal $Z$ down to $1\times1$
blocks (since a real vector can't be a genuine eigenvector for a complex eigenvalue), so the
$2\times2$ block is exactly the accommodation Schur's theorem makes to stay entirely within real
arithmetic when eigendecomposition's stricter "diagonal, using eigenvectors as the basis" demand
would otherwise force complex numbers everywhere.

</details>

---

## 📚 Further Reading
- Trefethen & Bau, *Numerical Linear Algebra*, Lectures 7-8 (QR) and 23 (Cholesky)
- Kingma & Welling, ["Auto-Encoding Variational Bayes"](https://arxiv.org/abs/1312.6114) (the reparameterization trick)

---

*Next notebook: [Tensor Operations](09_tensor_operations.ipynb)*